In [2]:
import os
import getpass

MODEL = "openai/gpt-6-luna"


In [3]:
if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")


---
## Contents

1. Saving and loading prompts as files
2. Managing prompts with LangSmith

---

# 1. Saving and Loading Prompts as Files

A **prompt template** is a prompt with blanks in it. The blanks are called **placeholders** and are written in curly braces, such as `{question}`. Your code fills them in each time it runs.

As a project grows, it helps to keep prompts out of your Python code. In their own file, prompts are easier to read, edit and share. LangChain does this with **serialisation**: turning an object into a standard text format that can be saved to a file and rebuilt later. The format used here is **JSON**, a plain-text format of named fields (such as `"template": "..."`) that is also common in web APIs.

In this section you will:

1. Save a prompt template to a JSON file.
2. Look at what is inside the file.
3. Load the prompt back and use it.

## Saving a prompt

The next cell creates a prompt template with one placeholder, `{question}`, and saves it with `.save()`. The file records:

- the **template text**;
- the **input variables**, the list of placeholders the code must fill in (`question`);
- the **template format**, the placeholder style used (`f-string`, the `{name}` style Python uses in f-strings).

> **Note:** The path `../../data/saved/` is relative to the folder this notebook is in (`notebooks/Prompt Engineering/`), so it points to the `data/saved` folder at the repository root. If that folder does not exist, the save will fail with a `FileNotFoundError`. Create the folder first.
>
> Saving to a local file is simple and works offline. When a team needs to share, version and monitor prompts, LangSmith (section 2) is the more common choice.

In [4]:
from langchain_core.prompts import PromptTemplate

template = """You are an insightful question answering bot. A user will submit \
questions, which are delimited by triple backticks, and you should respond in \
an insightful manner.

Question: ```{question}```

Think step by step before answering.

Answer:
"""

prompt = PromptTemplate(template=template, input_variables=["question"])
prompt.save("../../data/saved/cot_prompt.json")

/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/var/folders/vl/m6g01gp54cl4409b678xyrkm0000gn/T/ipykernel_62266/3459598564.py:15: LangChainDeprecationWarning: The method `BasePromptTemplate.save` was deprecated in langchain-core 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  prompt.save("../../data/saved/cot_prompt.json")


In [5]:
!cat ../../data/saved/cot_prompt.json

{
    "name": null,
    "input_variables": [
        "question"
    ],
    "optional_variables": [],
    "output_parser": null,
    "partial_variables": {},
    "metadata": null,
    "tags": null,
    "template": "You are an insightful question answering bot. A user will submit questions, which are delimited by triple backticks, and you should respond in an insightful manner.\n\nQuestion: ```{question}```\n\nThink step by step before answering.\n\nAnswer:\n",
    "template_format": "f-string",
    "validate_template": false,
    "_type": "prompt"
}

## Reading the saved file

Open `cot_prompt.json` (the output above) and look at the fields:

- `template` holds the prompt text.
- `input_variables` lists the placeholders the code must fill in (here, `question`).
- `template_format` tells LangChain how to read the placeholders.

**Watch out:** several lines in the prompt end with a backslash (`\`). In Python, a backslash at the end of a line inside a string joins that line to the next one. If there is no space before the backslash, the words run together, such as `submitquestions`. The code above puts a space before each backslash, so the words stay separate. This is a Python detail, not a LangChain bug.

## Loading a saved prompt

`load_prompt` reads the file and rebuilds the prompt, so you only need the file path. It checks the `_type` field in the file to decide what to build. The loaded prompt works in the same way as the original.

In [6]:
# All prompts are loaded through the `load_prompt` function.
from langchain_core.prompts import load_prompt

In [7]:
prompt = load_prompt("../../data/saved/cot_prompt.json")

/var/folders/vl/m6g01gp54cl4409b678xyrkm0000gn/T/ipykernel_62266/3953491361.py:1: LangChainDeprecationWarning: The function `load_prompt` was deprecated in LangChain 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  prompt = load_prompt("../../data/saved/cot_prompt.json")


In [8]:
print(prompt.template) # type: ignore # this line is to avoid mypy error

You are an insightful question answering bot. A user will submit questions, which are delimited by triple backticks, and you should respond in an insightful manner.

Question: ```{question}```

Think step by step before answering.

Answer:



## Section 1 summary

Files are simple. They need no account and no internet connection. The trade-offs are that someone must copy or sync the files between machines, and the files do not record which version of a prompt produced which output, unless you also use a version-control tool such as Git. Section 2 looks at a hosted option that addresses both problems.

# 2. Managing Prompts with LangSmith

**LangSmith** is a hosted service from LangChain. Its **prompt hub** is a shared place to store prompts with their full history. Each saved change becomes a new **commit** with a unique ID called a **commit hash**, a short code such as `c9839f14`. This works much like commits in Git.

A hosted hub helps in three ways:

- **Shared:** teammates load the same prompt without copying files around.
- **Versioned:** every change is recorded, and any earlier version can be loaded by its commit hash.
- **Pinned:** an app can load one exact version, so a change made elsewhere does not silently change its behaviour.

### Specifying a prompt version

Add a colon and a commit hash (or a commit tag) after the prompt name:

```python
from langsmith import Client

client = Client()
client.pull_prompt(f"{prompt_repo}:{version}")
```

In this snippet, `prompt_repo` and `version` are placeholders. Replace them with a prompt name and a real commit hash.

To load a public prompt shared by someone else, put their handle before the name, for example `efriis/my-first-prompt`.

### What you need before you start

- A LangSmith account and an API key for your workspace. Create one in LangSmith under **Settings > API Keys**. The next cell asks for the key with `getpass`, so it is not saved in the notebook.
- The notebook pulls the example prompt `prompt-ai-code-writer`. It must exist in your LangSmith workspace or be shared with you. If you get a not-found error, ask your lecturer to share it.
- Official setup guide: [LangSmith documentation](https://docs.langchain.com/langsmith/home)

In [9]:
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
if "LANGSMITH_API_KEY" not in os.environ:
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("LangSmith API Key:")


**Further reading:** [Managing and accessing prompts in your code](https://docs.langchain.com/langsmith/manage-prompts-programmatically)

In [10]:
from langsmith import Client

client = Client(api_url=os.environ["LANGSMITH_ENDPOINT"])

**Push** means upload to LangSmith. The cell below creates the prompt `joke-generator` if it does not exist, or updates it if it does.

> **Note:** Each run pushes the prompt again and adds to its version history. Only re-run this cell when you want a new version.

In [13]:
# Pushing a prompt to LangSmith 

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
url = client.push_prompt("joke-generator2", object=prompt)
# url is a link to the prompt in the UI
print(url)

https://eu.smith.langchain.com/prompts/joke-generator2/b9244726?organizationId=cc91df9b-ee35-4327-97aa-aa60f6e768cf


## Loading a specific version

1. **Latest by default:** `client.pull_prompt("name")` loads the most recent version. The SDK caches pulled prompts for 5 minutes by default, so a new version can take a few minutes to appear. Add `skip_cache=True` to fetch the latest version directly.
2. **Pinned version:** `client.pull_prompt("name:commit-hash")` loads that exact version. It does not change when the prompt is updated later.

The cells below load the latest version. To pin one, change the name to `prompt-ai-code-writer:<commit-hash>`. The hash `c9839f14` in the example is only an illustration. Replace it with a real hash from the prompt's version history in LangSmith.

## Using a pulled prompt in a chain

A **chain** links steps together, so the output of one step becomes the input of the next. In the cell below:

- `prompt` fills the `{question}` placeholder with the question you give it.
- `llm` is the language model. It is called through OpenRouter, using the `MODEL` value from the first cell.
- `prompt | llm` passes the filled-in prompt to the model. The `|` is LangChain's pipe operator, part of **LCEL** (LangChain Expression Language).
- `.invoke({...})` runs the chain with values for the placeholders.

The result is a message object. The cell after it prints `.content`, which is the plain text of the answer.

> **Note:** The cells below pull `prompt-ai-code-writer`. If you get a not-found error, the prompt is not in your workspace. Ask your lecturer to share it with you.

In [ ]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(model=MODEL)

prompt = client.pull_prompt("prompt-ai-code-writer")


# using LCEL
chain = prompt | llm

print(chain.invoke({"question": "How do I implement a transformer model using Hugging Face's Transformers library?"}).content)

AIMessage(content='Most applications use a pretrained Transformer and fine-tune it, rather than building attention layers from scratch. For example, this fine-tunes BERT for binary sentence classification:\n\n```bash\npip install transformers datasets accelerate\n```\n\n```python\nfrom datasets import load_dataset\nfrom transformers import (\n    AutoTokenizer,\n    AutoModelForSequenceClassification,\n    Trainer,\n    TrainingArguments,\n)\n\nmodel_name = "bert-base-uncased"\n\n# Load data and matching tokenizer/model\ndataset = load_dataset("glue", "mrpc")\ntokenizer = AutoTokenizer.from_pretrained(model_name)\n\ndef tokenize(batch):\n    return tokenizer(\n        batch["sentence1"],\n        batch["sentence2"],\n        truncation=True,\n        padding="max_length",\n        max_length=128,\n    )\n\nencoded = dataset.map(tokenize, batched=True)\n\nmodel = AutoModelForSequenceClassification.from_pretrained(\n    model_name,\n    num_labels=2,\n)\n\n# Configure training\nargs = Tr

## Pulling a prompt with its model

A prompt in LangSmith can also store the model it was written for. With `include_model=True`, `pull_prompt` returns the prompt and the model together as one chain (a `RunnableSequence`), ready to call with `.invoke`.

This is convenient, but there is a trade-off: the model is chosen by whoever saved the prompt, not by your code. The stored model may be an OpenAI model, which needs an `OPENAI_API_KEY` as well as your OpenRouter key. The next cell asks for that key if it is missing.

In [21]:

chain = client.pull_prompt("prompt-ai-code-writer", include_model=True, secrets_from_env=True)
chain.invoke({"question": "How do I implement a transformer model using Hugging Face's Transformers library?"})

/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/load/load.py:822: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  loaded_obj = {k: _load(v) for k, v in obj.items()}


AIMessage(content=[{'id': 'rs_tmp_c3whc7nn6aa', 'summary': [], 'type': 'reasoning', 'content': [{'text': "The user wants to know how to implement a transformer model using Hugging Face's Transformers library.\nI need to provide a concise, professor-like explanation covering the main steps: installation, loading a pre-trained model/tokenizer, inference, fine-tuning (high level), and saving.", 'type': 'reasoning_text'}], 'status': 'completed', 'format': 'unknown'}, {'type': 'text', 'text': 'Good. Let\'s keep this practical. There are three levels of "implementation" depending on your goal.\n\n### 1. Inference (Using a Pre-trained Model)\nThis is 90% of use cases. Don\'t reinvent the wheel.\n\n```python\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification\nimport torch\n\n# 1. Load tokenizer & model (downloads weights on first run)\nmodel_name = "distilbert-base-uncased-finetuned-sst-2-english"\ntokenizer = AutoTokenizer.from_pretrained(model_name)\nmodel = AutoMode

Failed to refresh cache entry prompt-ai-code-writer:with_model: Connection error caused failure to GET /commits/-/prompt-ai-code-writer/latest?include_model=true in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'eu.api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/-/prompt-ai-code-writer/latest?include_model=true (Caused by NameResolutionError("HTTPSConnection(host=\'eu.api.smith.langchain.com\', port=443): Failed to resolve \'eu.api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: None
API Key: lsv2_********************************************02
Failed to refresh cache entry prompt-ai-code-writer: Connection error caused failure to GET /commits/-/prompt-ai-code-writer/latest in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'eu.api.smith.langchain.com\', port=443): Max retries 

## Key takeaways

- A prompt behaves like code: a small change in wording can change the output. Store prompts in files or a hub, and track their versions.
- Pin the exact prompt version your app uses, so changes are deliberate. Test a new version before relying on it.
- Pinning a prompt version does not pin the model behind it. Model providers update their models, so the same prompt can behave differently over time. Record the model name alongside the prompt.
- Model output can differ between runs, even with the same input. Check the results yourself rather than trusting them.
- Be careful what you store. Prompts can be shared with your team, or made public. Never put API keys, personal data or confidential material in a prompt.